In [1]:
import os
import glob
import pandas as pd
import random
import numpy as np
import shutil

In [2]:
def get_df(folder):
    files = glob.glob(os.path.join(folder, '*.png'))
    df = pd.DataFrame(files, columns=['filepath'])
    df['platename'] = df.filepath.apply(lambda x: '_'.join(os.path.basename(x).split('_')[:-1]))
    df['year'] = df.filepath.apply(lambda x: os.path.basename(x).split('_')[0])
    df['location'] = df.filepath.apply(lambda x: os.path.basename(x).split('_')[1])
    df['week'] = df.filepath.apply(lambda x: os.path.basename(x).split('_')[2])

    return df

In [3]:
def split_column_with_simulated_annealing(df, column, targets, max_iter=10000, temp=1000, cooling_rate=0.003, random_seed=None):
    """
    Splits a DataFrame into subsets based on unique values of a specified column using Simulated Annealing 
    to match target sample counts.

    Parameters:
    - df (pd.DataFrame): The input DataFrame.
    - column (str): Name of the column in df to base the splitting on.
    - targets (list): List of target sample counts for each subset.
    - max_iter (int): Maximum number of iterations for the simulated annealing algorithm.
    - temp (float): Initial temperature for simulated annealing.
    - cooling_rate (float): Cooling rate for simulated annealing.
    - random_seed (int): Seed for random number generator to ensure reproducibility.

    Returns:
    - df_with_subsets (pd.DataFrame): Original DataFrame with an added 'subset' column indicating subset assignment.
    - assignment_summary (pd.DataFrame): Summary of assignments, actual vs. target samples, and deviations.
    """

    # Set random seed for reproducibility
    if random_seed is not None:
        random.seed(random_seed)
        np.random.seed(random_seed)

    # Calculate samples per unique value in the specified column
    value_sample_dict = df[column].value_counts().to_dict()
    unique_values = list(value_sample_dict.keys())

    # Objective function: Minimize deviation
    def calculate_deviation(assignment):
        subset_sums = [0] * len(targets)
        for val, subset_idx in assignment.items():
            subset_sums[subset_idx] += value_sample_dict[val]
        deviation = sum(abs(subset_sum - target) for subset_sum, target in zip(subset_sums, targets))
        return deviation, subset_sums

    # Simulated Annealing Algorithm
    def simulated_annealing():
        num_subsets = len(targets)
        current_assignment = {val: random.randint(0, num_subsets - 1) for val in unique_values}
        current_deviation, _ = calculate_deviation(current_assignment)
        best_assignment = current_assignment.copy()
        best_deviation = current_deviation

        local_temp = temp

        for i in range(max_iter):
            local_temp *= (1 - cooling_rate)
            new_assignment = current_assignment.copy()
            val_to_move = random.choice(unique_values)
            new_subset = random.randint(0, num_subsets - 1)
            new_assignment[val_to_move] = new_subset

            new_deviation, _ = calculate_deviation(new_assignment)
            delta = new_deviation - current_deviation

            if delta < 0 or random.uniform(0, 1) < np.exp(-delta / local_temp):
                current_assignment = new_assignment
                current_deviation = new_deviation
                if new_deviation < best_deviation:
                    best_assignment = new_assignment
                    best_deviation = new_deviation

        _, final_sums = calculate_deviation(best_assignment)
        return best_assignment, final_sums, best_deviation

    # Run simulated annealing
    best_assignment, final_sums, best_deviation = simulated_annealing()

    # Prepare results
    result = {}
    for i in range(len(targets)):
        result[f'Subset {i+1}'] = [val for val, idx in best_assignment.items() if idx == i]

    assignment_summary = []
    for i, (subset, values_in_subset) in enumerate(result.items()):
        assignment_summary.append({
            'Subset': f'Subset {i+1}',
            'Values': ', '.join(values_in_subset),
            'Actual Samples': final_sums[i],
            'Target Samples': targets[i],
            'Deviation': abs(final_sums[i] - targets[i])
        })

    assignment_summary_df = pd.DataFrame(assignment_summary)

    # Add subset assignment to original DataFrame
    value_to_subset = {val: idx for val, idx in best_assignment.items()}
    df_with_subsets = df.copy()
    df_with_subsets['subset'] = df_with_subsets[column].map(value_to_subset)

    return df_with_subsets, assignment_summary_df


In [4]:
import pandas as pd
import numpy as np
import random

def handle_excess_samples(df_with_subsets, assignment_summary, targets, strategy="remove", random_seed=None):
    """
    Handles excess samples in subsets beyond the target counts by keeping only the required number of samples.

    Parameters:
    - df_with_subsets (pd.DataFrame): DataFrame with subset assignments.
    - assignment_summary (pd.DataFrame): Summary with actual vs. target samples.
    - targets (list): List of target sample counts for each subset.
    - strategy (str): "remove" to keep only target samples, "mark" to flag excess ones.
    - random_seed (int): Seed for reproducibility.

    Returns:
    - df_adjusted (pd.DataFrame): DataFrame with adjusted or marked excess samples.
    """

    if random_seed is not None:
        random.seed(random_seed)
        np.random.seed(random_seed)

    # Create a deep copy to avoid modifying the original DataFrame
    df_adjusted = df_with_subsets.copy()

    # Ensure correct target mapping by pairing subset indices with targets
    unique_subsets = sorted(df_adjusted['subset'].unique())
    subset_target_mapping = dict(zip(unique_subsets, targets[:len(unique_subsets)]))

    # Process each subset separately
    subset_dfs = []  # Store processed subsets

    for subset_idx, target in subset_target_mapping.items():
        subset_df = df_adjusted[df_adjusted['subset'] == subset_idx].copy()  # Work on a copy

        current_count = len(subset_df)

        if current_count > target:
            excess_count = current_count - target
            #print(f"Subset {subset_idx}: Keeping {target} samples (target: {target}, current: {current_count})")

            if strategy == "remove":
                # Instead of dropping, randomly select `target` samples to KEEP
                subset_df = subset_df.sample(n=target, random_state=random_seed)

            elif strategy == "mark":
                # Mark excess samples instead of removing
                mark_indices = subset_df.sample(n=excess_count, random_state=random_seed).index
                subset_df.loc[mark_indices, 'excess_sample'] = True

            else:
                raise ValueError("Invalid strategy. Use 'remove' or 'mark'.")

        # Append cleaned subset to list
        subset_dfs.append(subset_df)

    # Recombine all cleaned subsets
    df_adjusted = pd.concat(subset_dfs, ignore_index=True)

    return df_adjusted


In [5]:
def get_df_subsets(df, targets, random_seed=42):
    df_with_subsets, assignment_summary = split_column_with_simulated_annealing(df, 'platename', targets, random_seed=random_seed)
    if assignment_summary['Deviation'].sum() > 0:
        return handle_excess_samples(df_with_subsets, assignment_summary, targets, strategy="remove", random_seed=random_seed), assignment_summary
    return df_with_subsets, assignment_summary

In [6]:
def calculate_custom_splits(total_samples, train_composition):
    """
    Calculates dataset splits based on custom rules:
    - Only 'good' and 'mislabel' samples are counted in total_samples.
    - 'other' samples are excluded from total_samples but train_composition may include them.
    - Train set has the given composition for 'good' and 'mislabel'.
    - Validation and test sets have 12.5% as many 'good' samples as the train set.
    - Validation set mimics the train ratio for 'good' and 'mislabel'; test is only 'good'.
    
    Returns:
        dict: sample counts per subset and type, with total_used (excluding 'other')
    """
    # Normalize over only 'good' and 'mislabel'
    gm_total = train_composition['good'] + train_composition['mislabel']
    good_prop = train_composition['good'] / gm_total
    mislabel_prop = train_composition['mislabel'] / gm_total

    # Calculate train size (T) based on desired val/test composition
    val_size = 0.125 / good_prop
    test_size = 0.125
    total_multiplier = 1 + val_size + test_size

    T = round(total_samples / total_multiplier)  # Train set size (good + mislabel)

    train_counts = {
        'good': round(good_prop * T),
        'mislabel': round(mislabel_prop * T),
        'other': 0  # optionally add this if needed elsewhere
    }

    val_good = round(0.125 * T)
    val_fraction = val_good / good_prop
    val_counts = {
        'good': val_good,
        'mislabel': round(mislabel_prop * val_fraction),
        'other': 0
    }

    test_counts = {'good': val_good, 'mislabel': 0, 'other': 0}

    total_used = train_counts['good'] + train_counts['mislabel'] + \
                 val_counts['good'] + val_counts['mislabel'] + \
                 test_counts['good']

    return {
        'train': train_counts,
        'validation': val_counts,
        'test': test_counts,
        'total_used': total_used
    }


In [7]:
total_samples = 4200
train_composition = {'good': 0.60, 'mislabel': 0.20, 'other': 0.20}

split_result = calculate_custom_splits(total_samples, train_composition)
split_result['train']['other'] = split_result['train']['mislabel']
split_result['validation']['other'] = split_result['validation']['mislabel']

split_result, split_result['train']['other']



({'train': {'good': 2439, 'mislabel': 813, 'other': 813},
  'validation': {'good': 406, 'mislabel': 135, 'other': 135},
  'test': {'good': 406, 'mislabel': 0, 'other': 0},
  'total_used': 4199},
 813)

In [8]:
# root_dir = r'C:\Users\u0159868\Documents\repos\stickybugs-outliers\data\all_data'
root_dir = '/home/u0159868/Documents/repos/stickybugs-outliers/data/phoneboxdata'

# c_folder = os.path.join(root_dir, 'c')
wmv_folder = os.path.join(root_dir, 'wmv')
wrl_folder = os.path.join(root_dir, 'wrl')
trash_folder = os.path.join(root_dir, 'wswl')
# m_folder = os.path.join(root_dir, 'm')
# v_folder = os.path.join(root_dir, 'v')

# c_trash_folder = os.path.join(root_dir, 'trash', 'c_trash')
# wmv_trash_folder = os.path.join(root_dir, 'trash', 'wmv_trash')
# wrl_trash_folder = os.path.join(root_dir, 'trash', 'wrl_trash')
# sp_trash_folder = os.path.join(root_dir, 'trash', 'sp_trash')
# c_extra_trash_folder = os.path.join(root_dir, 'trash', 'c_extra_trash')
# wswl_trash_folder = os.path.join(root_dir, 'trash', 'wswl_trash')
# t_trash_folder = os.path.join(root_dir, 'trash', 't_trash')
# sw_trash_folder = os.path.join(root_dir, 'trash', 'sw_trash')

In [13]:
# df_c = get_df(c_folder)
df_wmv = get_df(wmv_folder)
df_wrl = get_df(wrl_folder)
df_trash = get_df(trash_folder)
# df_m = get_df(m_folder)
# df_v = get_df(v_folder)

# df_c_trash = get_df(c_trash_folder)
# df_wmv_trash = get_df(wmv_trash_folder)
# df_wrl_trash = get_df(wrl_trash_folder)
# df_sp_trash = get_df(sp_trash_folder)
# df_c_extra_trash = get_df(c_extra_trash_folder)
# df_wswl_trash = get_df(wswl_trash_folder)
# df_t_trash = get_df(t_trash_folder)
# df_sw_trash = get_df(sw_trash_folder)

In [63]:
# Concatenate trash dataframes
# df_trash = pd.concat([df_c_trash, df_wmv_trash, df_wrl_trash, df_sp_trash, df_c_extra_trash, df_wswl_trash, df_t_trash, df_sw_trash])

In [10]:
pd.set_option('display.max_rows', None)

In [14]:
# automatic split
# df_c_subsets, assignment_summary_c = get_df_subsets(df_c, [
#     split_result['train']['good'], 
#     split_result['train']['mislabel'], 
#     split_result['test']['good'], 
#     split_result['validation']['good'], 
#     split_result['validation']['mislabel']
#     ])
df_wmv_subsets, assignment_summary_wmv = get_df_subsets(df_wmv, [
    split_result['train']['good'], 
    split_result['train']['mislabel'], 
    split_result['test']['good'], 
    split_result['validation']['good'], 
    split_result['validation']['mislabel']
    ])
df_wrl_subsets, assignment_summary_wrl = get_df_subsets(df_wrl, [
    split_result['train']['good'], 
    split_result['train']['mislabel'], 
    split_result['test']['good'], 
    split_result['validation']['good'], 
    split_result['validation']['mislabel']
    ])
df_trash_subsets, assignment_summary_trash = get_df_subsets(df_trash, [
    split_result['train']['other'], 
    split_result['train']['other'], 
    split_result['validation']['other'], 
    split_result['validation']['other'], 
    ])

# df_m_subsets, assignment_summary_m = get_df_subsets(df_m, [split_result['train']['good'], 219, 219, 205, 34], random_seed=423)
# df_v_subsets, assignment_summary_v = get_df_subsets(df_v, [
#     split_result['train']['good'], 
#     split_result['train']['mislabel'], 
#     split_result['test']['good'], 
#     split_result['validation']['good'], 
#     split_result['validation']['mislabel']
#     ], random_seed=423)
# df_trash_subsets, assignment_summary_trash = get_df_subsets(df_trash, [
#     split_result['train']['other'], 
#     split_result['train']['other'], 
#     split_result['train']['other'], 
#     split_result['validation']['other'], 
#     split_result['validation']['other'], 
#     split_result['validation']['other']
#     ])

In [15]:
assignment_summary_trash

,Subset,Values,Actual Samples,Target Samples,Deviation
0,Subset 1,"2024_herent_w39_witloof_ARTH5, 2024_herent_w43...",814,813,1
1,Subset 2,"2025_herent_w13_witloof_ARTFH19, 2024_herent_w...",813,813,0
2,Subset 3,"2024_herent_w02_witloof_ARTFH2, 2025_herent_w1...",137,135,2
3,Subset 4,"2024_herent_w41_witloof_ARTFH7bis, 2024_zwevez...",153,135,18


In [16]:
# print(df_c_subsets['subset'].value_counts())
print(df_wmv_subsets['subset'].value_counts())
print(df_wrl_subsets['subset'].value_counts())
print(df_trash_subsets['subset'].value_counts())

#print(df_m_subsets['subset'].value_counts())
# print(df_v_subsets['subset'].value_counts())
# print(df_trash_subsets['subset'].value_counts())

subset
0    2439
1     813
2     406
3     406
4     135
Name: count, dtype: int64
subset
0    2439
1     813
2     406
3     406
4     135
Name: count, dtype: int64
subset
0    813
1    813
2    135
3    135
Name: count, dtype: int64


In [30]:
# # 75 12.5 12.5 split
# df_c_subsets, assignment_summary_c = get_df_subsets(df_c, [1315, 219, 219, 205, 34])
# df_wmv_subsets, assignment_summary_wmv = get_df_subsets(df_wmv, [1315, 219, 219, 205, 34])
# df_m_subsets, assignment_summary_m = get_df_subsets(df_m, [1315, 219, 219, 205, 34], random_seed=423)
# df_v_subsets, assignment_summary_v = get_df_subsets(df_v, [1315, 219, 219, 205, 34], random_seed=423)
# df_trash_subsets, assignment_summary_trash = get_df_subsets(df_trash, [219, 219, 219, 34, 34, 34])

In [135]:
# # 60 20 20 split

# df_c_subsets, assignment_summary_c = get_df_subsets(df_c, [2791, 930, 500, 930, 310])
# df_wmv_subsets, assignment_summary_wmv = get_df_subsets(df_wmv, [2791, 930, 500, 930, 310])
# df_trash_subsets, assignment_summary_trash = get_df_subsets(df_trash, [930, 930, 310, 310])

In [ ]:
# 78 11 11 split

# df_c_subsets, assignment_summary_c = get_df_subsets(df_c, [3768, 539, 539, 539, 77])
# df_wmv_subsets, assignment_summary_wmv = get_df_subsets(df_wmv, [3768, 539, 539, 539, 77])
# df_c_trash_subsets, assignment_summary_c_trash = get_df_subsets(df_c_trash, [539, 77])
# df_wmv_trash_subsets, assignment_summary_wmv_trash = get_df_subsets(df_wmv_trash, [539, 77])

In [17]:
# root_dir_dest = r'C:\Users\u0159868\Documents\repos\stickybugs-outliers\data\all_data_split_75_12.5_12.5'
root_dir_dest = '/home/u0159868/Documents/repos/stickybugs-outliers/data/all_data_split_60_20_20'
wmv_dest_folders = ['train/wmv/wmv_good', 'train/wrl/wmv_for_wrl', 'test/wmv', 'val/wmv/wmv_good', 'val/wrl/wmv_for_wrl']
wrl_dest_folders = ['train/wrl/wrl_good', 'train/wmv/wrl_for_wmv', 'test/wrl', 'val/wrl/wrl_good', 'val/wmv/wrl_for_wmv']
trash_dest_folders = ['train/wmv/wmv_trash', 'train/wrl/wrl_trash', 'val/wmv/wmv_trash', 'val/wrl/wrl_trash']
# c_dest_folders = ['train/c/c_good', 'train/wmv/c_for_wmv', 'test/c', 'val/c/c_good', 'val/wmv/c_for_wmv']
# wmv_dest_folders = ['train/wmv/wmv_good', 'train/c/wmv_for_c', 'test/wmv', 'val/wmv/wmv_good', 'val/c/wmv_for_c']
# m_dest_folders = ['train/m/m_good', 'train/wmv/m_for_wmv', 'test/m', 'val/m/m_good', 'val/wmv/m_for_wmv']
# v_dest_folders = ['train/v/v_good', 'train/wmv/v_for_wmv', 'test/v', 'val/v/v_good', 'val/wmv/v_for_wmv']
# trash_dest_folders = ['train/c/c_trash', 'train/wmv/wmv_trash', 'train/v/v_trash', 'val/c/c_trash', 'val/wmv/wmv_trash', 'val/v/v_trash']

In [18]:
def copy_files_to_dest(df_subsets, root_dir_dest, dest_folders):
    for i in range(len(dest_folders)):
        dest_path = os.path.join(root_dir_dest, dest_folders[i])
        os.makedirs(dest_path, exist_ok=True)
        df_subset_i = df_subsets[df_subsets['subset'] == i]
        df_subset_i['dest_filepath'] = df_subset_i.filepath.apply(lambda x: os.path.join(dest_path, os.path.basename(x)))

        # Copy file from filepath to dest_filepath using shutil.copy
        for _, row in df_subset_i.iterrows():
            shutil.copy2(row['filepath'], row['dest_filepath'])

In [19]:
# copy_files_to_dest(df_c_subsets, root_dir_dest, c_dest_folders)
copy_files_to_dest(df_wmv_subsets, root_dir_dest, wmv_dest_folders)
copy_files_to_dest(df_wrl_subsets, root_dir_dest, wrl_dest_folders)
copy_files_to_dest(df_trash_subsets, root_dir_dest, trash_dest_folders)

# copy_files_to_dest(df_m_subsets, root_dir_dest, m_dest_folders)
# copy_files_to_dest(df_v_subsets, root_dir_dest, v_dest_folders)
# copy_files_to_dest(df_trash_subsets, root_dir_dest, trash_dest_folders)

/tmp/ipykernel_3301068/2475360090.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_subset_i['dest_filepath'] = df_subset_i.filepath.apply(lambda x: os.path.join(dest_path, os.path.basename(x)))
/tmp/ipykernel_3301068/2475360090.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_subset_i['dest_filepath'] = df_subset_i.filepath.apply(lambda x: os.path.join(dest_path, os.path.basename(x)))
/tmp/ipykernel_3301068/2475360090.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slic

In [ ]:
# def copy_folder_contents(src_folder, dest_folder):
#     """
#     Copies all files from the source folder to the destination folder.

#     Parameters:
#         src_folder (str): Path to the source folder.
#         dest_folder (str): Path to the destination folder.
#     """
#     os.makedirs(dest_folder, exist_ok=True)
#     for file in os.listdir(src_folder):
#         shutil.copy2(os.path.join(src_folder, file), os.path.join(dest_folder, file))


In [73]:
def copy_insect_class_folders(root_dir, former_class, dest_class, dataset_types):
    """
    Copies files from a specific folder of one insect class to another insect class for multiple dataset types.

    Parameters:
        root_dir (str): Root directory containing the dataset.
        former_class (str): The former insect class (e.g., 'c').
        dest_class (str): The destination insect class (e.g., 'm').
        dataset_types (list): List of dataset types (e.g., ['train', 'val']).
    """
    for dataset_type in dataset_types:
        # Construct source and destination folder names dynamically
        src_folder = os.path.join(root_dir, dataset_type, former_class, f'wmv_for_{former_class}')
        dest_folder = os.path.join(root_dir, dataset_type, dest_class, f'wmv_for_{dest_class}')
        
        # Ensure the destination folder exists
        os.makedirs(dest_folder, exist_ok=True)
        
        # Copy files from source to destination
        for file in os.listdir(src_folder):
            shutil.copy2(os.path.join(src_folder, file), os.path.join(dest_folder, file))

In [74]:
dataset_types = ['train', 'val']

#copy_insect_class_folders(root_dir_dest, 'c', 'm', dataset_types)
copy_insect_class_folders(root_dir_dest, 'c', 'v', dataset_types)

In [ ]:
# wmv_for_c_folder_train = os.path.join(root_dir_dest, 'train/c/wmv_for_c')
# wmv_for_m_folder_train = os.path.join(root_dir_dest, 'train/m/wmv_for_m')

# wmv_for_c_folder_val = os.path.join(root_dir_dest, 'val/c/wmv_for_c')
# wmv_for_m_folder_val = os.path.join(root_dir_dest, 'val/m/wmv_for_m')

# copy_folder_contents(wmv_for_c_folder_train, wmv_for_m_folder_train)
# copy_folder_contents(wmv_for_c_folder_val, wmv_for_m_folder_val)

In [129]:
# Example usage after running the split_column_with_simulated_annealing function

# Assuming df_with_subsets and assignment_summary are already created
targets = [539, 77]

# Option 1: Remove excess samples to exactly meet targets
df_cleaned = handle_excess_samples(df_with_subsets, assignment_summary, targets, strategy="remove", random_seed=42)

# Option 2: Mark excess samples for later handling
df_marked = handle_excess_samples(df_with_subsets, assignment_summary, targets, strategy="mark", random_seed=42)

# Check results
print("Cleaned DataFrame (Excess Removed):")
print(df_cleaned['subset'].value_counts())

print("\nMarked DataFrame (Excess Flagged):")
print(df_marked['excess_sample'].value_counts())


Cleaned DataFrame (Excess Removed):
0    539
1     77
Name: subset, dtype: int64

Marked DataFrame (Excess Flagged):
False    616
True      31
Name: excess_sample, dtype: int64


In [118]:
df_0_platenames = df_with_subsets[df_with_subsets.subset == 0].platename.unique()
df_1_platenames = df_with_subsets[df_with_subsets.subset == 1].platename.unique()
df_2_platenames = df_with_subsets[df_with_subsets.subset == 2].platename.unique()
df_3_platenames = df_with_subsets[df_with_subsets.subset == 3].platename.unique()
df_4_platenames = df_with_subsets[df_with_subsets.subset == 4].platename.unique()

# Check if there are any common platename between subsets
common_platenames_01 = set(df_0_platenames).intersection(df_1_platenames)
common_platenames_02 = set(df_0_platenames).intersection(df_2_platenames)
common_platenames_03 = set(df_0_platenames).intersection(df_3_platenames)
common_platenames_04 = set(df_0_platenames).intersection(df_4_platenames)
common_platenames_12 = set(df_1_platenames).intersection(df_2_platenames)
common_platenames_13 = set(df_1_platenames).intersection(df_3_platenames)
common_platenames_14 = set(df_1_platenames).intersection(df_4_platenames)
common_platenames_23 = set(df_2_platenames).intersection(df_3_platenames)
common_platenames_24 = set(df_2_platenames).intersection(df_4_platenames)
common_platenames_34 = set(df_3_platenames).intersection(df_4_platenames)
common_platenames_01, common_platenames_02, common_platenames_03, common_platenames_04, common_platenames_12, common_platenames_13, common_platenames_14, common_platenames_23, common_platenames_24, common_platenames_34

(set(), set(), set(), set(), set(), set(), set(), set(), set(), set())